<a href="https://colab.research.google.com/github/SOKMRP/People_Counting_Assistant/blob/main/CA1_T3_Team2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
ULTRALYTICS_VERSION = "8.4.120"

try:
    import google.colab  # noqa: F401
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION],
                   check=True)
if RUNNING_ON_COLAB:
    import subprocess
    # roboflow is left unpinned on purpose. It talks to a hosted service, so the
    # version that matches the service is whichever one is current.
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "roboflow"],
        check=True,
    )

import json
import os


from ultralytics import YOLO

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389"
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))


WORK_DIR = os.path.join(PROJECT_ROOT, "runs", "Ca1")
os.makedirs(WORK_DIR, exist_ok=True)




Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [11]:

import re
from getpass import getpass

# Paste the download code from Roboflow between the triple quotes, then delete the
# key so the line reads api_key="". Leave it empty to use the sample set.
ROBOFLOW_CODE = """!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="")
project = rf.workspace("c389006-team-2").project("people-counting-assistant-sona0")
version = project.version(8)
dataset = version.download("yolov8")


"""

SAMPLE_DIR = os.path.join(PROJECT_ROOT, "data", "sample", "walkway")
YOLO_FORMATS = ("yolov8", "yolo26", "yolov11", "yolov12", "yolov9", "yolov5")


def read_roboflow_code(code):
    """Read the workspace, project, version and format out of Roboflow's code.

    Returns None when nothing was pasted. Raises ValueError when the code still
    holds the private key, or lacks a value the download needs.
    """
    if not code.strip():
        return None
    key = re.search(r"api_key\s*=\s*[\"']([^\"']*)[\"']", code)
    if key and key.group(1).strip():
        raise ValueError(
            "The pasted code still holds your private API key. Delete it so the "
            "line reads api_key=\"\", then run the cell again. The key is asked "
            "for when the cell runs, so it is never saved in the notebook.")
    found = {}
    for name, pattern in (("workspace", r"\.workspace\(\s*[\"']([^\"']+)[\"']"),
                          ("project", r"\.project\(\s*[\"']([^\"']+)[\"']"),
                          ("version", r"\.version\(\s*(\d+)\s*\)"),
                          ("format", r"\.download\(\s*[\"']([^\"']+)[\"']")):
        match = re.search(pattern, code)
        if not match:
            raise ValueError(
                "No %s in the pasted code. Copy the whole block Roboflow shows "
                "under Show download code." % name)
        found[name] = match.group(1)
    found["version"] = int(found["version"])
    # YOLOv8, YOLO26 and the other YOLO text formats write the same files: one
    # label file per image and a data.yaml. Anything else is a different layout.
    if found["format"] not in YOLO_FORMATS:
        raise ValueError(
            "The code downloads the %r format. Choose YOLOv8 in the Download "
            "Dataset window and copy the code again." % found["format"])
    return found


def roboflow_key():
    """The private key from Colab Secrets, or else from a hidden prompt."""
    if RUNNING_ON_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("ROBOFLOW_API_KEY").strip()
        except Exception:
            pass  # no secret saved, or this notebook is not allowed to read it
    return getpass("Roboflow private API key: ").strip()


ROBOFLOW = read_roboflow_code(ROBOFLOW_CODE)

if ROBOFLOW:
    from roboflow import Roboflow

    print("workspace %(workspace)s, project %(project)s, version %(version)d" % ROBOFLOW)
    project = Roboflow(api_key=roboflow_key()).workspace(
        ROBOFLOW["workspace"]).project(ROBOFLOW["project"])
    export = project.version(ROBOFLOW["version"]).download(
        ROBOFLOW["format"], location=os.path.join(WORK_DIR, "dataset"), overwrite=True
    )
    DATASET_DIR = export.location
    USING_SAMPLE_DATA = False
else:
    if not os.path.isdir(SAMPLE_DIR):
        raise FileNotFoundError(
            "Nothing was pasted and there is no sample set at %s.\n"
            "Paste the Roboflow download code, or run scripts/make_sample_data.py."
            % SAMPLE_DIR
        )
    DATASET_DIR = SAMPLE_DIR
    USING_SAMPLE_DATA = True



DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")


if not os.path.exists(DATA_YAML):
    raise FileNotFoundError(
        "No dataset at %s. Copy data/sample from the repository to "
        "Drive, or run scripts/make_sample_data.py."
        % DATASET_DIR
    )

print("dataset :", DATASET_DIR)
print("work dir:", WORK_DIR)

workspace c389006-team-2, project people-counting-assistant-sona0, version 8
Roboflow private API key: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/C389/runs/Ca1/dataset in yolov8:: 100%|██████████| 940/940 [04:42<00:00,  3.33it/s]

dataset : /content/drive/MyDrive/C389/runs/Ca1/dataset
work dir: /content/drive/MyDrive/C389/runs/Ca1


In [12]:


import yaml

def read_class_names(yaml_path):
    """Read the ordered class list out of a data.yaml."""
    with open(yaml_path, encoding="utf-8") as handle:
        data = yaml.safe_load(handle)
    names = data.get("names")
    if isinstance(names, list):
        return [str(name).strip() for name in names]
    if isinstance(names, dict):
        return [str(names[k]).strip() for k in sorted(names.keys())]
    raise ValueError("No names: line in %s" % yaml_path)


BASELINE_CLASSES = ["person_left", "person_right"]
DATASET_CLASSES = read_class_names(DATA_YAML)

for class_id, name in enumerate(DATASET_CLASSES):
    print("%d  %s" % (class_id, name))

0  person_left
1  person_right
2  phone_using_left
3  phone_using_right


In [13]:
def normalise_data_yaml(dataset_dir):
    """Rewrite data.yaml with an absolute path, keeping the class list.

    Returns the path to the file that was written.
    """
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    names_line = "names: " + str(read_class_names(yaml_path))

    splits = []
    for key, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
        if os.path.isdir(os.path.join(dataset_dir, folder, "images")):
            splits.append("%s: %s/images" % (key, folder))

    if not any(line.startswith("train:") for line in splits):
        raise FileNotFoundError(
            "No train/images folder under %s. The export did not unpack the way "
            "this notebook expects." % dataset_dir)

    class_count = len(read_class_names(yaml_path))
    with open(yaml_path, "w", encoding="utf-8") as handle:
        print("# Paths rewritten by this notebook so that they resolve from",
              "anywhere.", file=handle)
        print("path:", dataset_dir.replace(os.sep, "/"), file=handle)
        for line in splits:
            print(line, file=handle)
        print("", file=handle)
        print("nc:", class_count, file=handle)
        print(names_line, file=handle)
    return yaml_path


normalise_data_yaml(DATASET_DIR)
print(open(DATA_YAML, encoding="utf-8").read())


# Paths rewritten by this notebook so that they resolve from anywhere.
path: /content/drive/MyDrive/C389/runs/Ca1/dataset
train: train/images
val: valid/images
test: test/images

nc: 4
names: ['person_left', 'person_right', 'phone_using_left', 'phone_using_right']



In [14]:
def extend_class_list(existing_classes, new_classes):
    """Add classes to a class list without disturbing the existing IDs.

    Args:
        existing_classes: the ordered class list already in use
        new_classes: names to add

    Returns:
        a new ordered list, existing classes first and unchanged

    Requirement:
        Every existing class keeps the position it had. New names go on the end,
        in the order given, and a name already present is not added twice.
    """
    extended = list(existing_classes)
    for name in new_classes:
        if name not in extended:
            extended.append(name)
    return extended

In [15]:
NEW_CLASSES = [name for name in DATASET_CLASSES if name not in BASELINE_CLASSES]
CLASS_NAMES = extend_class_list(BASELINE_CLASSES, NEW_CLASSES)

print("baseline :", BASELINE_CLASSES)
print("new      :", NEW_CLASSES)
print("extended :", CLASS_NAMES)

# The check that would have caught an insert. It compares positions, not
# membership, because a list can hold the same names in the wrong order.
if CLASS_NAMES[:len(BASELINE_CLASSES)] != BASELINE_CLASSES:
    raise ValueError(
        "The original classes moved. Every label file written in Lesson 2 now "
        "means something different. Append, never insert."
    )
if CLASS_NAMES != DATASET_CLASSES:
    raise ValueError(
        "The extended list %s does not match the dataset's own list %s. The "
        "annotation and the training config have to agree."
        % (CLASS_NAMES, DATASET_CLASSES)
    )
print("")
print("Positions 0 and 1 are unchanged. Every Lesson 2 label still means what it did.")

baseline : ['person_left', 'person_right']
new      : ['phone_using_left', 'phone_using_right']
extended : ['person_left', 'person_right', 'phone_using_left', 'phone_using_right']

Positions 0 and 1 are unchanged. Every Lesson 2 label still means what it did.


In [16]:
def instances_per_class(dataset_dir, split, class_names):
    """Count annotated instances of every class in one split.

    Args:
        dataset_dir: the dataset folder, holding train, valid and test
        split: "train", "valid" or "test"
        class_names: the ordered class list

    Returns:
        dict mapping every class name to a whole number, zeros included

    Requirement:
        Count boxes, not files. One image holding three people is three
        instances of the relevant classes and one image.
    """
    counts = {}
    for name in class_names:
        counts[name] = 0

    label_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.isdir(label_dir):
        return counts

    for filename in sorted(os.listdir(label_dir)):
        if not filename.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, filename), encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                class_id = int(line.split()[0])
                if class_id < len(class_names):
                    counts[class_names[class_id]] += 1
    return counts

In [17]:
SPLITS = ["train", "valid", "test"]
INSTANCE_COUNTS = {split: instances_per_class(DATASET_DIR, split, CLASS_NAMES)
                   for split in SPLITS}

print("%-20s %7s %7s %7s %7s" % ("class", "train", "valid", "test", "total"))
for name in CLASS_NAMES:
    row = [INSTANCE_COUNTS[split][name] for split in SPLITS]
    print("%-20s %7d %7d %7d %7d" % (name, row[0], row[1], row[2], sum(row)))

for name in NEW_CLASSES:
    if INSTANCE_COUNTS["valid"][name] == 0:
        print("")
        print("WARNING: %s has no instances in validation. Its score will be "
              "reported and will mean nothing." % name)

class                  train   valid    test   total
person_left              313      43      44     400
person_right             261      46      38     345
phone_using_left         117      15       6     138
phone_using_right        122      11       7     140


In [18]:
MODEL_VARIANT = "yolo26n.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
FLIP_LEFT_RIGHT = 0.0   # still 0.0, and for the same reason as Lesson 3
RUN_NAME = "Ca1_team2_T3"

# model = YOLO(MODEL_VARIANT)
# model.train(
#     data=DATA_YAML,
#     epochs=EPOCHS,
#     imgsz=IMAGE_SIZE,
#     batch=BATCH_SIZE,
#     fliplr=FLIP_LEFT_RIGHT,
#     project=WORK_DIR,
#     name=RUN_NAME,
#     exist_ok=True,
#     seed=0,
# )

RUN_DIR = os.path.join(WORK_DIR, RUN_NAME)
BEST_WEIGHTS = os.path.join(RUN_DIR, "weights", "best.pt")
print("weights:", BEST_WEIGHTS)

weights: /content/drive/MyDrive/C389/runs/Ca1/Ca1_team2_T3/weights/best.pt


In [19]:
trained = YOLO(BEST_WEIGHTS)
metrics = trained.val(data=DATA_YAML, split="test", verbose=False)

PER_CLASS = {}
for index, class_id in enumerate(metrics.ap_class_index):
    PER_CLASS[CLASS_NAMES[int(class_id)]] = {
        "precision": float(metrics.box.p[index]),
        "recall": float(metrics.box.r[index]),
        "ap50": float(metrics.box.ap50[index]),
    }

print("%-20s %9s %7s %8s %9s" % ("class", "precision", "recall", "mAP@.5", "instances"))
for name in CLASS_NAMES:
    if name in PER_CLASS:
        row = PER_CLASS[name]
        total = sum(INSTANCE_COUNTS[split][name] for split in SPLITS)
        print("%-20s %9.3f %7.3f %8.3f %9d"
              % (name, row["precision"], row["recall"], row["ap50"], total))

Ultralytics 8.4.120 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 51.5±5.9 MB/s, size: 50.2 KB)
val: Scanning /content/drive/MyDrive/C389/runs/Ca1/dataset/test/labels.cache... 47 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 47/47 6.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.2it/s 2.5s
                   all         47         95      0.785      0.847      0.852      0.611
Speed: 4.4ms preprocess, 14.0ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val
class                precision  recall   mAP@.5 instances
person_left              0.799   0.815    0.895       400
person_right             0.857   0.789    0.856       345
phone_using_left         0.793   0.833    0.835       138
phone_using_right        0.

In [20]:
import shutil
CONFIDENCE_THRESHOLD = 0.4
ARTIFACT_DIR = os.path.join(PROJECT_ROOT, "artifacts", "Ca1")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

shutil.copy(BEST_WEIGHTS, os.path.join(ARTIFACT_DIR, "best.pt"))

with open(os.path.join(ARTIFACT_DIR, "Ca1_01_metrics.json"), "w",
          encoding="utf-8") as handle:
    json.dump({
        "classes": CLASS_NAMES,
        "per_class": PER_CLASS,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "config": {"model_variant": MODEL_VARIANT, "epochs": EPOCHS,
                   "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
                   "fliplr": FLIP_LEFT_RIGHT},
        "used_sample_data": USING_SAMPLE_DATA,
    }, handle, indent=1)

print("best.pt and baseline_metrics.json saved to", ARTIFACT_DIR)
print("Lesson 4 loads the weights from there. Lesson 5 compares against the metrics.")

best.pt and baseline_metrics.json saved to /content/drive/MyDrive/C389/artifacts/Ca1
Lesson 4 loads the weights from there. Lesson 5 compares against the metrics.


In [21]:
import io
import os
import time

import gradio as gr
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

print("gradio:", gr.__version__)

if RUNNING_ON_COLAB:
    import subprocess
    # gradio is left unpinned, because the page only needs a recent version. The
    # version this prints is the one the page was built with.
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio"],
        check=True,
    )

if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/C389"
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

WEIGHTS = os.path.join(PROJECT_ROOT, "artifacts", "Ca1", "best.pt")
print("weights:", WEIGHTS)

gradio: 6.31.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
weights: /content/drive/MyDrive/C389/artifacts/Ca1/best.pt


In [22]:
if os.path.exists(WEIGHTS):
    MODEL_PATH = WEIGHTS
    USING_TRAINED_MODEL = True
else:
    MODEL_PATH = "yolo26n.pt"
    USING_TRAINED_MODEL = False
    print("WARNING: no best.pt at %s" % WEIGHTS)
    print("Falling back to the stock model. It cannot report direction.")
    print("Re-run the Lesson 3 notebook to the end before submitting anything.")

In [23]:
MODEL = YOLO(MODEL_PATH)
CLASS_NAMES = [MODEL.names[key] for key in sorted(MODEL.names)]

CONFIDENCE_THRESHOLD = 0.40
MAX_IMAGE_PIXELS = 40_000_000
SERVICE_VERSION = "1.0.0"

print("model loaded from  :", MODEL_PATH)
print("classes            :", CLASS_NAMES)
print("confidence threshold:", CONFIDENCE_THRESHOLD)

model loaded from  : /content/drive/MyDrive/C389/artifacts/Ca1/best.pt
classes            : ['person_left', 'person_right', 'phone_using_left', 'phone_using_right']
confidence threshold: 0.4


In [24]:
def validate_image(uploaded):
    """Check that what arrived can be turned into an image to run on.

    Args:
        uploaded: whatever the caller sent. A path, a file like object, a PIL
            image, or None when nothing was sent at all

    Returns:
        (image, error), exactly one of which is None. The error is a dict
        carrying a status code and a message the caller could act on

    Requirement:
        Four cases are refused before any inference runs: nothing sent, an empty
        file, a file that is not a readable image, and an image so large it would
        exhaust memory.

        A service that hands a text file to the model and returns a stack trace
        has told the caller nothing. Every refusal here names what was wrong and
        what to send instead.
    """
    if uploaded is None:
        return None, {"status": 400, "error": "No image was sent.",
                      "detail": "Attach a JPEG or PNG and send it again."}

    try:
        if isinstance(uploaded, Image.Image):
            image = uploaded
        elif isinstance(uploaded, (bytes, bytearray)):
            if not uploaded:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file carried no data."}
            image = Image.open(io.BytesIO(uploaded))
            image.load()
        else:
            if os.path.getsize(uploaded) == 0:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file '%s' is zero bytes."
                                        % os.path.basename(str(uploaded))}
            image = Image.open(uploaded)
            image.load()
    except (UnidentifiedImageError, OSError):
        return None, {"status": 415, "error": "That file is not a readable image.",
                      "detail": "Send a JPEG or PNG."}

    if image.width * image.height > MAX_IMAGE_PIXELS:
        return None, {"status": 413, "error": "The image is too large to process.",
                      "detail": "%d by %d pixels is beyond the limit of %d pixels."
                                % (image.width, image.height, MAX_IMAGE_PIXELS)}

    return image.convert("RGB"), None

In [25]:
def count_people(image, threshold):
    """Run the model on one image and total the detections by class.

    Args:
        image: a PIL image, already validated
        threshold: detections below this confidence are dropped

    Returns:
        (counts, kept), where counts maps every class name to a whole number

    Requirement:
        The same rule as Lesson 3. Every class appears in counts, including the
        ones at zero, and an image with nobody in it is a valid result of zero
        rather than an error.
    """
    result = MODEL.predict(image, conf=0.01, verbose=False)[0]

    kept = []
    boxes = result.boxes
    if boxes is not None:
        for index in range(len(boxes)):
            confidence = round(float(boxes.conf[index].item()), 3)
            if confidence < threshold:
                continue
            class_id = int(boxes.cls[index].item())
            kept.append({
                "class_name": CLASS_NAMES[class_id],
                "confidence": confidence,
                "box": [round(float(v), 1) for v in boxes.xyxy[index].tolist()],
            })

    counts = {}
    for name in CLASS_NAMES:
        counts[name] = 0
    for detection in kept:
        counts[detection["class_name"]] = counts[detection["class_name"]] + 1

    return counts, kept

In [26]:
def build_reply(uploaded, threshold=CONFIDENCE_THRESHOLD):
    """The whole service, in one function. Validate, count, answer.

    Returns a dict either way. A caller never has to tell success from failure by
    reading prose: a reply carrying "error" failed, and one carrying "counts" did
    not.

    A CA1 twist that adds a class or an extra field changes this function and
    nothing else, which is why the reply is built in one place.
    """
    image, error = validate_image(uploaded)
    if error is not None:
        return error, None

    started = time.time()
    counts, detections = count_people(image, threshold)
    elapsed = time.time() - started

    reply = {
        "status": 200,
        "counts": counts,
        "confidence_threshold": threshold,
        "model_version": SERVICE_VERSION,
        "trained_model": USING_TRAINED_MODEL,
        "image_size": [image.width, image.height],
        "inference_seconds": round(elapsed, 3),
        "detections": detections,
    }
    # Printed rather than kept, so the Colab output below the cell is this
    # service's log. Section 6 reads it.
    print("counted %s in %.3fs on a %dx%d image"
          % (counts, elapsed, image.width, image.height))
    return reply, image

In [27]:
def annotate(image, detections):
    """Draw the kept boxes, so the reply can be checked by eye as well as read."""
    from PIL import ImageDraw

    marked = image.copy()
    draw = ImageDraw.Draw(marked)
    for detection in detections:
        left, top, right, bottom = detection["box"]
        colour = (214, 69, 65) if detection["class_name"].endswith("left") else (46, 112, 196)
        draw.rectangle([left, top, right, bottom], outline=colour, width=3)
        draw.text((left + 3, max(0, top - 12)),
                  "%s %.2f" % (detection["class_name"], detection["confidence"]),
                  fill=colour)
    return marked


def predict(uploaded, threshold):
    """What the page calls. Returns the annotated image and the reply."""
    reply, image = build_reply(uploaded, threshold)
    if image is None:
        return None, reply
    return annotate(image, reply["detections"]), reply

In [33]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil", label="Walkway photograph"),
        gr.Slider(0.05, 0.95, value=CONFIDENCE_THRESHOLD, step=0.05,
                  label="Confidence threshold"),
    ],
    outputs=[
        gr.Image(label="Detections"),
        gr.JSON(label="Reply"),
    ],
    title="People Counting Assistant",
    description="Upload one photograph. The reply says how many people are "
                "moving left and how many are moving right.",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1b756270a163c6b4c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:
import tempfile

scratch = tempfile.mkdtemp()

text_file = os.path.join(scratch, "not_an_image.txt")
with open(text_file, "w", encoding="utf-8") as handle:
    handle.write("this is not a photograph")

empty_file = os.path.join(scratch, "empty.jpg")
open(empty_file, "wb").close()

huge = Image.new("RGB", (9000, 7000), (200, 200, 200))
huge_file = os.path.join(scratch, "huge.jpg")
huge.save(huge_file, quality=30)

nobody = Image.new("RGB", (640, 480), (214, 210, 202))
nobody_file = os.path.join(scratch, "nobody.jpg")
nobody.save(nobody_file)

CASES = [
    ("nothing sent", None),
    ("a text file", text_file),
    ("an empty file", empty_file),
    ("a very large image", huge_file),
    ("an image with nobody in it", nobody_file),
]

# Collected first, printed afterwards. The service logs a line for every call it
# serves, and a library may print a one-time message the first time it is asked
# for something, so printing the table as it goes puts that chatter through the
# middle of the thing being read.
rows = []
for label, payload in CASES:
    reply, _ = build_reply(payload)
    summary = reply.get("error") or ("counts " + str(reply.get("counts")))
    rows.append((label, reply["status"], summary))

print("")
print("%-28s %-7s %s" % ("case", "status", "what came back"))
for label, status, summary in rows:
    print("%-28s %-7s %s" % (label, status, summary))

counted {'person_left': 0, 'person_right': 0, 'phone_using_left': 0, 'phone_using_right': 0} in 0.019s on a 640x480 image

case                         status  what came back
nothing sent                 400     No image was sent.
a text file                  415     That file is not a readable image.
an empty file                400     The upload was empty.
a very large image           413     The image is too large to process.
an image with nobody in it   200     counts {'person_left': 0, 'person_right': 0, 'phone_using_left': 0, 'phone_using_right': 0}


In [22]:
# Warm, measured here, where this process is exactly the warm service.
sample = Image.new("RGB", (640, 480), (214, 210, 202))

warm = []
for attempt in range(5):
    started = time.time()
    count_people(sample, CONFIDENCE_THRESHOLD)
    warm.append(time.time() - started)

warm_average = sum(warm) / len(warm)
print("warm inference, average : %.3f s" % warm_average)

warm inference, average : 0.017 s


In [23]:
import subprocess
import sys

# A separate process, so nothing this notebook has already done counts. It
# imports, loads the weights and answers one request, and reports where its own
# time went. The figure that matters is the total measured from out here, because
# that is what the caller waits through.
weights = os.path.abspath(MODEL_PATH) if os.path.exists(MODEL_PATH) else MODEL_PATH
probe_path = "/content/cold_start_probe.py"

with open(probe_path, "w", encoding="utf-8") as handle:
    print("import sys, time", file=handle)
    print("launch = float(sys.argv[1])", file=handle)
    print("start = time.time()", file=handle)
    print("from ultralytics import YOLO", file=handle)
    print("from PIL import Image", file=handle)
    print("imported = time.time()", file=handle)
    print("model = YOLO(%r)" % weights, file=handle)
    print("loaded = time.time()", file=handle)
    print("model.predict(Image.new('RGB', (640, 480)), conf=0.01, verbose=False)",
          file=handle)
    print("answered = time.time()", file=handle)
    print("print('  importing the libraries : %.3f s' % (imported - start))", file=handle)
    print("print('  reading the weights     : %.3f s' % (loaded - imported))", file=handle)
    print("print('  answering the first time: %.3f s' % (answered - loaded))", file=handle)
    print("print('ANSWERED %.3f' % (answered - launch))", file=handle)

started = time.time()
probe = subprocess.run([sys.executable, probe_path, repr(started)],
                       capture_output=True, text=True)
process_lifetime = time.time() - started

# The caller waits from the moment the process is launched until it answers. What
# happens after that, releasing the GPU and shutting down, can take seconds and
# nobody is waiting for it, so timing the whole subprocess overstates the wait.
cold_total = process_lifetime
report = []
for line in probe.stdout.splitlines():
    if not line.strip():
        continue
    if line.startswith("ANSWERED "):
        cold_total = float(line.split()[1])
    else:
        report.append(line)

if probe.returncode != 0:
    print("The probe did not run. Its last words:")
    print(probe.stderr.strip()[-600:])
else:
    print("inside a process that has just started:")
    for line in report:
        print(line)
    print("")
    print("somebody arriving at a cold service waits : %.3f s" % cold_total)
    print("somebody arriving at a warm one waits     : %.3f s" % warm_average)
    print("cold is %.0f times the warm figure" % (cold_total / warm_average))
    print("")
    print("the probe process lived for %.3f s in total. The difference is it"
          % process_lifetime)
    print("shutting down and releasing the GPU, which no caller waits for.")

inside a process that has just started:
  importing the libraries : 2.209 s
  reading the weights     : 0.092 s
  answering the first time: 0.903 s

somebody arriving at a cold service waits : 3.252 s
somebody arriving at a warm one waits     : 0.017 s
cold is 194 times the warm figure

the probe process lived for 6.637 s in total. The difference is it
shutting down and releasing the GPU, which no caller waits for.


In [32]:
demo.close()
print("The share link is closed.")

Closing server running on port: 7860
The share link is closed.
